
# Chorale Code


## A. Import Intervals and Other Code

* The first step is to import all the code required for the Notebook
* **`arrow/run`** or **`Shift + Enter`** in the following cell:

In [1]:
import crim_intervals
from crim_intervals import * 
from crim_intervals import main_objs
import crim_intervals.visualizations as viz
import pandas as pd
import re
import altair as alt
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact
from pandas.io.json import json_normalize
from pyvis.network import Network
from IPython.display import display
import requests
import os
import glob as glob


MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    print("created folder : ", MYDIR)
else:
    print(MYDIR, "folder already exists.")
    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)
else:
    print(MUSDIR, "folder already exists.")

saved_csv folder already exists.
Music_Files folder already exists.


## Importing Files

#### Import One File

In [5]:
# Select a prefix:

prefix = 'Music_Files/'
# prefix = 'https://crimproject.org/mei/'
# Add your filename here

mei_file = '255_cleaned.xml'

# join the strings and import piece
url = prefix + mei_file
piece = importScore(url)

print(piece.metadata)


{'title': 'ach_gott_und_herr', 'composer': 'Bach', 'date': None}


#### Import Corpus of Local Files

In [24]:

corpus_list = []
for name in glob.glob('Music_Files/*'):
    corpus_list.append(name)
corpus = CorpusBase(corpus_list)

In [59]:
# define the first function
func1 = ImportedPiece.sonorities

# now run the corpus through that function
# the arguments for the function are set as kwwargs (a dictionary of keyword arguments)
# we set the metadata as False for the first step
list_of_sonority_dfs = corpus.batch(func=func1, 
                           kwargs={'compound': True}, 
                           metadata=False)


# and now add measure/beat information 
func2 = ImportedPiece.detailIndex
list_of_sonority_dfs_det = corpus.batch(func=func2, 
                                      kwargs={'df': list_of_sonority_dfs}, 
                                      metadata=False)

list_of_sonority_dfs_det
# now make ngrams and add the metadata
func3 = ImportedPiece.ngrams
list_of_sonority_ngrams_dfs = corpus.batch(func=func3, 
                                      kwargs={'n': 2, 'df': list_of_sonority_dfs_det}, 
                                      metadata=True)
list_of_sonority_ngrams_dfs

# concatenate the results
complete_results = pd.concat(list_of_sonority_ngrams_dfs)
complete_results.reset_index(inplace=True)

complete_results

/Users/rfreedma/opt/anaconda3/envs/encoding_music/lib/python3.9/site-packages/crim_intervals/main_objs.py:821: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ret.dropna(inplace=True, how='all')
/Users/rfreedma/opt/anaconda3/envs/encoding_music/lib/python3.9/site-packages/pandas/core/frame.py:5582: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  return super().sort_index(
/Users/rfreedma/opt/anaconda3/envs/encoding_music/lib/python3.9/site-packages/crim_intervals/main_objs.py:821: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas

,Measure,Beat,Sonority,Composer,Title,Date
0,0,1.00,"(15/12/10, 8/5/10)",Bach,ach_gott_und_herr,None
1,1,1.00,"(8/5/10, 8/12/10)",Bach,ach_gott_und_herr,None
2,1,2.00,"(8/12/10, 7/12/10)",Bach,ach_gott_und_herr,None
3,1,2.50,"(7/12/10, 15/12/10)",Bach,ach_gott_und_herr,None
4,1,3.00,"(15/12/10, 8/12/10)",Bach,ach_gott_und_herr,None
...,...,...,...,...,...,...
153,19,4.50,"(5/12/10, 13/12/10)",Bach,wie_schon_leuchtet,None
154,20,1.00,"(13/12/10, 5/12/10)",Bach,wie_schon_leuchtet,None
155,20,2.00,"(5/12/10, 4/12/10)",Bach,wie_schon_leuchtet,None
156,20,2.25,"(4/12/10, 5/12/10)",Bach,wie_schon_leuchtet,None


## Count Ngrams

In [60]:


counted = complete_results['Sonority'].value_counts().to_frame()
counted

,Sonority
"(8/12/10, 8/5/10)",4
"(7/12/10, 15/12/10)",4
"(15/12/10, 8/5/10)",3
"(8/6/3, 8/5/10)",3
"(15/13/10, 9/14/12)",3
...,...
"(8/5/10, 8/5/11)",1
"(8/15/10, 8/5/10)",1
"(6/3/2, 6/3)",1
"(8/5/10, 8/5/10)",1


## Grouped by Ngram

Note that the rows where the 'title' column have more than one value are the pieces that can be 'paired' in a Network!

In [61]:
grouped = complete_results.groupby('Sonority').agg({'Composer': 'unique', 'Title': 'unique'})
grouped.head(40)

,Composer,Title
Sonority,,
"(, 6/13/10)",[Bach],[wie_schon_leuchtet]
"(13/12/10, 5/12/10)",[Bach],[wie_schon_leuchtet]
"(13/12/10, 8/12/10)",[Bach],[ach_gott_und_herr]
"(15/12/10, )",[Bach],[wie_schon_leuchtet]
"(15/12/10, 6/13/10)",[Bach],[ach_gott_und_herr]
"(15/12/10, 8/12/10)",[Bach],[ach_gott_und_herr]
"(15/12/10, 8/3/12)",[Bach],[ach_gott_und_herr]
"(15/12/10, 8/5/10)",[Bach],"[ach_gott_und_herr, wie_schon_leuchtet]"
"(15/12/10, 8/5/3)",[Bach],[wie_schon_leuchtet]
